# LeetCode 707 — Design Linked List bằng Python

Link đề: https://leetcode.com/problems/design-linked-list/ · Độ khó: **Medium** · Chủ đề: Design, Linked List

Skeleton LeetCode (Python):

```python
class MyLinkedList:

    def __init__(self):
        ...

    def get(self, index: int) -> int:
        ...

    def addAtHead(self, val: int) -> None:
        ...

    def addAtTail(self, val: int) -> None:
        ...

    def addAtIndex(self, index: int, val: int) -> None:
        ...

    def deleteAtIndex(self, index: int) -> None:
        ...


# Your MyLinkedList object will be instantiated and called as such:
# obj = MyLinkedList()
# param_1 = obj.get(index)
# obj.addAtHead(val)
# obj.addAtTail(val)
# obj.addAtIndex(index,val)
# obj.deleteAtIndex(index)
```

Ghi chú: bài **thiết kế lớp**. Skeleton Python của LeetCode **không cho sẵn** lớp nút, nên ta tự định nghĩa `ListNode` ngay trong code (nếu nộp bài, đặt định nghĩa nút phía trên lớp `MyLinkedList`). Notebook tự dựng bộ chạy chuỗi lệnh `run_ops` mô phỏng cách LeetCode gọi lớp, và kiểm tra bằng một **oracle** là danh sách Python thông thường (chỉ để **đối chiếu**, không dùng để nộp vì đề cấm thư viện danh sách liên kết có sẵn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tự cài danh sách liên kết với năm thao tác `get`, `addAtHead`, `addAtTail`, `addAtIndex`, `deleteAtIndex`, trong đó chỉ số **bắt đầu từ 0** và chỉ số không hợp lệ được xử lý theo quy tắc riêng (`get` trả `-1`, thêm vào chỗ quá `độ dài` thì bỏ qua, xóa chỉ số không hợp lệ thì bỏ qua).
2. Nắm các công cụ Python: lớp `ListNode` với `val`/`next`, **tham chiếu** (hai tên cùng trỏ một đối tượng), `None`, vòng `for _ in range(k)` để **đi `k` bước** dọc danh sách, nút **canh gác (dummy/sentinel)**, nút có `prev` (danh sách hai chiều), và mô phỏng "con trỏ" bằng **chỉ số mảng**.
3. Đi từ **danh sách đơn không nút canh (nhiều trường hợp đặc biệt)** → **danh sách đơn + nút `dummy` + biến đếm `cnt`** → **danh sách đôi với hai nút canh, đi từ đầu gần hơn** → **danh sách liên kết cài trên mảng (next[] + danh sách ô trống)**.
4. Hiểu mấu chốt: mọi thao tác chèn/xóa đều quy về **"tìm nút đứng trước vị trí cần sửa" rồi nối lại hai con trỏ**; nút canh `dummy` biến "đầu danh sách" thành một trường hợp bình thường.
5. Tránh các lỗi kinh điển: quên cập nhật độ dài, nối con trỏ **sai thứ tự**, dùng `>=` thay cho `>` khi `index == độ dài`, quên xử lý xóa đầu danh sách (khi không có `dummy`), và quên sửa `prev` ở danh sách đôi.

## 2. Đề bài & ví dụ

Hãy tự cài một danh sách liên kết (đơn hoặc đôi). Mọi nút được đánh chỉ số **từ 0**. Lớp `MyLinkedList`:

- `MyLinkedList()`: khởi tạo.
- `get(index)`: trả về giá trị của nút thứ `index`; nếu `index` không hợp lệ trả `-1`.
- `addAtHead(val)`: thêm nút `val` **trước** nút đầu tiên; nút mới trở thành nút đầu.
- `addAtTail(val)`: thêm nút `val` làm nút **cuối**.
- `addAtIndex(index, val)`: thêm nút `val` **trước** nút thứ `index`. Nếu `index` **bằng độ dài** danh sách thì thêm vào cuối. Nếu `index` **lớn hơn** độ dài thì **không thêm**.
- `deleteAtIndex(index)`: xóa nút thứ `index` nếu `index` hợp lệ.

**Ví dụ:**

```
ops  = ["MyLinkedList", "addAtHead", "addAtTail", "addAtIndex", "get", "deleteAtIndex", "get"]
args = [[], [1], [3], [1, 2], [1], [1], [1]]
Output: [null, null, null, null, 2, null, 3]
```

Danh sách biến đổi: `[] → [1] → [1, 3] → [1, 2, 3]`; `get(1) = 2`; xóa chỉ số 1: `[1, 3]`; `get(1) = 3`.

**Ràng buộc:** `0 <= index, val <= 1000`; **không** dùng thư viện danh sách liên kết có sẵn; tối đa `2000` lệnh gọi.

In [1]:
import random
import time
from typing import List


def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế lớp: ops[0] là tên lớp (hàm tạo)."""
    obj, out = None, []
    for i, (op, a) in enumerate(zip(ops, args)):
        if i == 0:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


ops = ["MyLinkedList", "addAtHead", "addAtTail", "addAtIndex", "get", "deleteAtIndex", "get"]
args = [[], [1], [3], [1, 2], [1], [1], [1]]
EXPECTED = [None, None, None, None, 2, None, 3]
for o, a in zip(ops, args):
    print(f"{o:<14} {a}")
print("kết quả mong đợi:", EXPECTED)

MyLinkedList   []
addAtHead      [1]
addAtTail      [3]
addAtIndex     [1, 2]
get            [1]
deleteAtIndex  [1]
get            [1]
kết quả mong đợi: [None, None, None, None, 2, None, 3]


## 3. Kiến thức nền

### 3.1. Nút và danh sách liên kết đơn

Một **nút** (node) chứa một giá trị `val` và một **tham chiếu** `next` tới nút kế tiếp (hoặc `None` nếu là nút cuối). **Danh sách** chỉ cần giữ tham chiếu tới **nút đầu** (`head`); muốn đến nút thứ `k` ta phải đi **từng bước** qua `next`. Khác với `list` của Python (truy cập `a[k]` ngay), danh sách liên kết truy cập chỉ số `O(k)` nhưng chèn/xóa tại một nút đã biết là `O(1)`.

In [2]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


# 1 -> 2 -> 3
head = ListNode(1, ListNode(2, ListNode(3)))
cur = head
while cur:
    print(cur.val, end=" -> ")
    cur = cur.next
print("None")


def to_list(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


print(to_list(head))

1 -> 2 -> 3 -> None
[1, 2, 3]


### 3.2. Tham chiếu: hai tên, một đối tượng

`b = a` **không** sao chép nút; `a` và `b` là hai tên cho **cùng** một đối tượng. Gán `b.next = ...` làm thay đổi cái mà `a` cũng thấy. Đây là lý do ta có thể "đi dọc danh sách" bằng một biến `cur` rồi sửa `cur.next` để nối lại danh sách. Dùng `is` để kiểm tra **cùng một đối tượng**.

In [3]:
a = ListNode(10)
b = a
b.val = 99
print(a.val, a is b)
c = ListNode(99)
print(a == c, a is c)             # khác đối tượng (== mặc định cũng so theo danh tính)

99 True
False False


### 3.3. Đi `k` bước: `for _ in range(k)`

Để đứng ở nút thứ `k`, bắt đầu từ `head` rồi lặp `k` lần `cur = cur.next`. Biến `_` là quy ước "không dùng giá trị vòng lặp".

In [4]:
head = ListNode(5, ListNode(6, ListNode(7, ListNode(8))))
cur = head
for _ in range(2):
    cur = cur.next
print("nút thứ 2 có val =", cur.val)           # 7
cur = head
for _ in range(0):
    cur = cur.next
print("nút thứ 0 có val =", cur.val)           # 5 (đi 0 bước)

nút thứ 2 có val = 7
nút thứ 0 có val = 5


### 3.4. Chèn và xóa: nối lại con trỏ (quan trọng: **thứ tự**)

Chèn nút mới `x` **sau** nút `pre`: phải đặt `x.next = pre.next` **trước**, rồi mới `pre.next = x`. Nếu làm ngược (`pre.next = x` trước) thì ta đã mất tham chiếu tới phần đuôi cũ. Xóa nút sau `pre`: `pre.next = pre.next.next`.

In [5]:
head = ListNode(1, ListNode(3))
pre = head
x = ListNode(2)
x.next = pre.next            # bước 1: x trỏ vào phần đuôi cũ
pre.next = x                 # bước 2: pre trỏ vào x
print(to_list(head))         # [1, 2, 3]

pre.next = pre.next.next     # xóa nút sau pre (nút 2)
print(to_list(head))         # [1, 3]

# làm sai thứ tự:
head = ListNode(1, ListNode(3))
pre, x = head, ListNode(2)
pre.next = x
x.next = pre.next            # x.next trỏ chính x -> vòng lặp, mất nút 3
print(x.next is x)

[1, 2, 3]
[1, 3]
True


### 3.5. Nút canh (dummy / sentinel)

Chèn/xóa ở **đầu** danh sách khác chèn/xóa ở giữa vì **không có nút đứng trước**. Mẹo: thêm một nút giả `dummy` đứng **trước** nút đầu (`dummy.next = head`). Khi đó **mọi** vị trí đều có một nút đứng trước, và chỉ cần một đoạn code. Nút đầu thật là `dummy.next`.

In [6]:
dummy = ListNode()                      # val vô nghĩa
dummy.next = ListNode(1, ListNode(2))
pre = dummy                              # nút đứng trước vị trí 0 chính là dummy
pre.next = ListNode(0, pre.next)         # chèn vào đầu, không cần trường hợp đặc biệt
print(to_list(dummy.next))
pre.next = pre.next.next                 # xóa đầu
print(to_list(dummy.next))

[0, 1, 2]
[1, 2]


### 3.6. Danh sách đôi: thêm `prev`

Mỗi nút có thêm `prev` trỏ nút trước. Có hai nút canh `head` và `tail` ở hai đầu. Nhờ `prev` ta có thể **đi ngược** từ đuôi: muốn đến chỉ số ở nửa sau, đi từ đuôi nhanh hơn. Đổi lại, mỗi lần chèn/xóa phải sửa **bốn** con trỏ (`next` và `prev` của hai nút kề).

In [7]:
class DNode:
    def __init__(self, val=0, prev=None, next=None):
        self.val, self.prev, self.next = val, prev, next


h, t = DNode(), DNode()
h.next, t.prev = t, h
x = DNode(7, h, t)                       # chèn 7 giữa h và t
h.next = x                               # sửa next của nút trước
t.prev = x                               # sửa prev của nút sau
print(h.next.val, t.prev.val, x.prev is h, x.next is t)

7 7 True True


### 3.7. Mô phỏng con trỏ bằng chỉ số mảng

Ở ngôn ngữ không có tham chiếu (hoặc khi cần bộ nhớ liền kề), ta dùng hai mảng `val[]` và `nxt[]`: "nút" là một **chỉ số ô**, `nxt[i]` là chỉ số ô kế tiếp (`-1` là hết). Ô bị xóa được đưa vào một **danh sách ô trống** (`free`) để dùng lại.

In [8]:
val = [0, 10, 20, 30]        # ô 0 là nút canh
nxt = [1, 2, 3, -1]          # 0 -> 1 -> 2 -> 3
i, out = nxt[0], []
while i != -1:
    out.append(val[i])
    i = nxt[i]
print(out)

[10, 20, 30]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy theo dõi **Ví dụ** bằng hình vẽ (danh sách cùng chỉ số):

| Lệnh | Danh sách sau lệnh | Ghi chú |
|---|---|---|
| (khởi tạo) | `[]` | rỗng |
| `addAtHead(1)` | `[1]` | nút đầu mới |
| `addAtTail(3)` | `[1, 3]` | đi đến cuối, nối thêm |
| `addAtIndex(1, 2)` | `[1, 2, 3]` | chèn **trước** nút chỉ số 1 (nút `3`), tức **sau** nút chỉ số 0 |
| `get(1)` | `[1, 2, 3]` | đi 1 bước từ đầu → `2` |
| `deleteAtIndex(1)` | `[1, 3]` | nối nút `1` thẳng tới nút `3` |
| `get(1)` | `[1, 3]` | → `3` |

**Quy luật chung:** để sửa tại chỉ số `i` ta cần **nút đứng ngay trước vị trí `i`** (gọi là `pre`, chỉ số `i − 1`). Chèn: `new.next = pre.next; pre.next = new`. Xóa: `pre.next = pre.next.next`. Nút trước vị trí `0` không tồn tại — chính là chỗ nút canh `dummy` phát huy tác dụng.

Điều kiện hợp lệ: `get`/`delete` cần `0 <= index < độ dài`; `add` cần `0 <= index <= độ dài` (bằng `độ dài` nghĩa là thêm vào cuối).

## 5. Cách 1 — Brute Force / baseline: danh sách đơn **không** nút canh

**Ý tưởng:** giữ `head` (có thể `None`) và `size`. Mỗi thao tác tự đi đến đúng nút, **xử lý riêng trường hợp đầu danh sách**:

- `get`: kiểm tra `0 <= index < size`, đi `index` bước từ `head`.
- `addAtHead`: nút mới trỏ vào `head` cũ, rồi `head = nút mới`.
- `addAtIndex(index, val)`: `index > size` thì bỏ qua; `index == 0` thì gọi `addAtHead`; còn lại đi `index − 1` bước tới `pre` rồi chèn sau `pre`.
- `addAtTail` = `addAtIndex(size, val)`.
- `deleteAtIndex`: `index == 0` thì `head = head.next`; còn lại đi tới `pre` rồi `pre.next = pre.next.next`.

Mỗi thao tác tốn `O(index)`. Đây là cách "trực tiếp" nhất, nhưng có **nhiều nhánh đặc biệt** (đầu danh sách) dễ sai.

In [9]:
class LinkedNoDummy:
    def __init__(self):
        self.head = None
        self.size = 0

    def get(self, index):
        if index < 0 or index >= self.size:
            return -1
        cur = self.head
        for _ in range(index):
            cur = cur.next
        return cur.val

    def addAtHead(self, val):
        self.head = ListNode(val, self.head)
        self.size += 1

    def addAtTail(self, val):
        self.addAtIndex(self.size, val)

    def addAtIndex(self, index, val):
        if index > self.size:
            return
        if index <= 0:
            self.addAtHead(val)
            return
        pre = self.head
        for _ in range(index - 1):
            pre = pre.next
        pre.next = ListNode(val, pre.next)
        self.size += 1

    def deleteAtIndex(self, index):
        if index < 0 or index >= self.size:
            return
        if index == 0:
            self.head = self.head.next
        else:
            pre = self.head
            for _ in range(index - 1):
                pre = pre.next
            pre.next = pre.next.next
        self.size -= 1

    def dump(self):                                   # chỉ dùng để test
        return to_list(self.head)


out = run_ops(LinkedNoDummy, ops, args)
print(out)
assert out == EXPECTED

[None, None, None, None, 2, None, 3]


**Độ phức tạp:** `get`, `addAtIndex`, `deleteAtIndex` đều `O(index)` ≤ `O(n)`; `addAtHead` `O(1)`; `addAtTail` `O(n)` (phải đi tới cuối). Space `O(n)`.

**Nút thắt:** đoạn code có **hai nhánh** cho mỗi thao tác sửa (đầu danh sách / còn lại), dễ quên một nhánh. Ta muốn một đoạn code duy nhất cho mọi vị trí: **nút canh**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề 1 — luôn quy về "nút đứng trước".** Mọi thao tác chèn/xóa tại chỉ số `i` chỉ cần nút `pre` ở chỉ số `i − 1`. Nếu có một nút canh `dummy` làm nút "chỉ số −1" thì `pre` luôn tồn tại với `0 <= i <= size`: đi `i` bước từ `dummy`.

**Bổ đề 2 — giữ độ dài `cnt`.** Duy trì `cnt` = số nút thật; kiểm tra chỉ số hợp lệ bằng `cnt` mà không cần đi hết danh sách để biết độ dài (và không cần dò `None` rủi ro).

**Bổ đề 3 — hai điều kiện biên khác nhau.** `get`/`delete` hợp lệ khi `index < cnt`; `add` hợp lệ khi `index <= cnt` (chèn vào cuối). Sai biên ở đây là lỗi số một của bài.

**Bổ đề 4 — có thể giảm bước đi.** Ở danh sách **đôi** có hai nút canh ở hai đầu, ta đi từ đầu gần hơn: tối đa `n/2` bước. Ở cài đặt **mảng**, "con trỏ" là chỉ số nên cùng thuật toán chạy được trên bộ nhớ liền kề.

Từ đó có các cách: **dummy + `cnt`** (Cách 2, cách nộp), **danh sách đôi hai nút canh** (Cách 3), và **danh sách trên mảng** (Cách 4).

## 7. Các cách tiếp theo

Tổng cộng 4 cách: Cách 1 (mục 5) và ba cách dưới đây. Cách 2 là **cải tiến cấu trúc** của Cách 1 (một đoạn code cho mọi vị trí); Cách 3 đổi **kiểu danh sách** (hai chiều + đi từ đầu gần hơn); Cách 4 đổi **cách biểu diễn con trỏ** (chỉ số mảng, tái sử dụng ô).

### Cách 2 — Danh sách đơn + nút `dummy` + biến đếm `cnt` (cách chính thức)

**Giải thích từng đoạn:**

1. `__init__`: `dummy = ListNode()`, `cnt = 0`.
2. `get(index)`: ngoài `[0, cnt)` trả `-1`; còn lại đi `index` bước từ `dummy.next`.
3. `addAtIndex(index, val)`: `index > cnt` thì bỏ qua; đi `index` bước từ `dummy` để tới `pre` (nút trước vị trí `index`); chèn `ListNode(val, pre.next)` rồi `pre.next = nút mới`; `cnt += 1`.
4. `addAtHead(val)` = `addAtIndex(0, val)`; `addAtTail(val)` = `addAtIndex(cnt, val)`.
5. `deleteAtIndex(index)`: ngoài `[0, cnt)` thì bỏ qua; đi tới `pre`; `pre.next = pre.next.next`; `cnt -= 1`.

In [10]:
class LinkedDummy:
    def __init__(self):
        self.dummy = ListNode()
        self.cnt = 0

    def get(self, index):
        if index < 0 or index >= self.cnt:
            return -1
        cur = self.dummy.next
        for _ in range(index):
            cur = cur.next
        return cur.val

    def addAtHead(self, val):
        self.addAtIndex(0, val)

    def addAtTail(self, val):
        self.addAtIndex(self.cnt, val)

    def addAtIndex(self, index, val):
        if index > self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        pre.next = ListNode(val, pre.next)          # ListNode(val, pre.next) nối đuôi trước, rồi pre.next = nút mới
        self.cnt += 1

    def deleteAtIndex(self, index):
        if index < 0 or index >= self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        t = pre.next
        pre.next = t.next
        t.next = None
        self.cnt -= 1

    def dump(self):
        return to_list(self.dummy.next)


out = run_ops(LinkedDummy, ops, args)
print(out)
assert out == EXPECTED

[None, None, None, None, 2, None, 3]


**Code nộp LeetCode (Cách 2):**

```python
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


class MyLinkedList:
    def __init__(self):
        self.dummy = ListNode()
        self.cnt = 0

    def get(self, index: int) -> int:
        if index < 0 or index >= self.cnt:
            return -1
        cur = self.dummy.next
        for _ in range(index):
            cur = cur.next
        return cur.val

    def addAtHead(self, val: int) -> None:
        self.addAtIndex(0, val)

    def addAtTail(self, val: int) -> None:
        self.addAtIndex(self.cnt, val)

    def addAtIndex(self, index: int, val: int) -> None:
        if index > self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        pre.next = ListNode(val, pre.next)
        self.cnt += 1

    def deleteAtIndex(self, index: int) -> None:
        if index < 0 or index >= self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        t = pre.next
        pre.next = t.next
        t.next = None
        self.cnt -= 1
```

**Độ phức tạp:** `get`, `addAtIndex`, `deleteAtIndex`: `O(index)`; `addAtHead`: `O(1)`; `addAtTail`: `O(n)`. Space `O(n)`.

### Cách 3 — Danh sách **đôi** với hai nút canh, đi từ đầu gần hơn

**Giải thích từng đoạn:**

1. Hai nút canh: `head` (trước nút đầu) và `tail` (sau nút cuối), nối `head.next = tail`, `tail.prev = head`. `size` là số nút thật.
2. Hàm phụ `_node_at(index)` trả về **nút thật thứ `index`** (với `index == size` trả về `tail`): nếu `index <= size // 2` đi **tới** từ `head.next`, ngược lại đi **lùi** từ `tail` đúng `size − index` bước.
3. `get`: `_node_at(index).val` nếu hợp lệ.
4. `addAtIndex(index, val)`: `nxt = _node_at(index)` (nút sẽ đứng **sau** nút mới), `pre = nxt.prev`; tạo nút mới nối giữa `pre` và `nxt`; sửa 4 con trỏ.
5. `deleteAtIndex`: `node = _node_at(index)`; `node.prev.next = node.next`; `node.next.prev = node.prev`.

Nhờ đó `addAtTail` còn `O(1)` (`index = size` ⇒ đi lùi 0 bước tới `tail`) và các thao tác tối đa `n/2` bước.

In [11]:
class LinkedDoubly:
    def __init__(self):
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head
        self.size = 0

    def _node_at(self, index):
        """Nút thật thứ index; index == size trả về nút canh tail."""
        if index <= self.size // 2:
            cur = self.head.next
            for _ in range(index):
                cur = cur.next
        else:
            cur = self.tail
            for _ in range(self.size - index):
                cur = cur.prev
        return cur

    def get(self, index):
        if index < 0 or index >= self.size:
            return -1
        return self._node_at(index).val

    def addAtHead(self, val):
        self.addAtIndex(0, val)

    def addAtTail(self, val):
        self.addAtIndex(self.size, val)

    def addAtIndex(self, index, val):
        if index > self.size:
            return
        nxt = self._node_at(index)
        pre = nxt.prev
        node = DNode(val, pre, nxt)
        pre.next = node
        nxt.prev = node
        self.size += 1

    def deleteAtIndex(self, index):
        if index < 0 or index >= self.size:
            return
        node = self._node_at(index)
        node.prev.next = node.next
        node.next.prev = node.prev
        node.prev = node.next = None
        self.size -= 1

    def dump(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.val)
            cur = cur.next
        return out


out = run_ops(LinkedDoubly, ops, args)
print(out)
assert out == EXPECTED

[None, None, None, None, 2, None, 3]


**Code nộp LeetCode (Cách 3):**

```python
class DNode:
    def __init__(self, val=0, prev=None, next=None):
        self.val, self.prev, self.next = val, prev, next


class MyLinkedList:
    def __init__(self):
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head
        self.size = 0

    def _node_at(self, index: int) -> DNode:
        if index <= self.size // 2:
            cur = self.head.next
            for _ in range(index):
                cur = cur.next
        else:
            cur = self.tail
            for _ in range(self.size - index):
                cur = cur.prev
        return cur

    def get(self, index: int) -> int:
        if index < 0 or index >= self.size:
            return -1
        return self._node_at(index).val

    def addAtHead(self, val: int) -> None:
        self.addAtIndex(0, val)

    def addAtTail(self, val: int) -> None:
        self.addAtIndex(self.size, val)

    def addAtIndex(self, index: int, val: int) -> None:
        if index > self.size:
            return
        nxt = self._node_at(index)
        pre = nxt.prev
        node = DNode(val, pre, nxt)
        pre.next = node
        nxt.prev = node
        self.size += 1

    def deleteAtIndex(self, index: int) -> None:
        if index < 0 or index >= self.size:
            return
        node = self._node_at(index)
        node.prev.next = node.next
        node.next.prev = node.prev
        self.size -= 1
```

**Độ phức tạp:** mỗi thao tác tối đa `n/2` bước: `O(n)` nhưng hằng số nhỏ hơn; `addAtHead`/`addAtTail` `O(1)`. Space `O(n)`.

### Cách 4 — Danh sách liên kết cài trên **mảng** (`val[]`, `nxt[]`, danh sách ô trống)

**Giải thích từng đoạn:**

1. Hai mảng song song `val` và `nxt`; **ô 0** là nút canh (đóng vai `dummy`), `nxt[0]` là ô đầu thật, `-1` là hết. `free` là ngăn xếp các ô đã xóa để **tái sử dụng**; `size` là số nút.
2. `_new(v, nx)`: lấy một ô từ `free` (hoặc nối thêm vào cuối mảng), gán `val`, `nxt`, trả về chỉ số ô.
3. Đi tới "nút trước" bằng `cur = nxt[cur]` lặp `index` lần từ ô 0; chèn: `nxt[pre] = _new(v, nxt[pre])`; xóa: `t = nxt[pre]; nxt[pre] = nxt[t]; free.append(t)`.

Cùng thuật toán với Cách 2, nhưng "con trỏ" là **chỉ số** — hữu ích khi ngôn ngữ không có tham chiếu hay khi cần cấp phát trước.

In [12]:
class LinkedArrayPool:
    def __init__(self):
        self.val = [0]                       # ô 0: nút canh
        self.nxt = [-1]
        self.free = []                       # các ô trống để tái sử dụng
        self.size = 0

    def _new(self, v, nx):
        if self.free:
            s = self.free.pop()
            self.val[s], self.nxt[s] = v, nx
        else:
            self.val.append(v)
            self.nxt.append(nx)
            s = len(self.val) - 1
        return s

    def get(self, index):
        if index < 0 or index >= self.size:
            return -1
        cur = self.nxt[0]
        for _ in range(index):
            cur = self.nxt[cur]
        return self.val[cur]

    def addAtHead(self, val):
        self.addAtIndex(0, val)

    def addAtTail(self, val):
        self.addAtIndex(self.size, val)

    def addAtIndex(self, index, val):
        if index > self.size:
            return
        pre = 0
        for _ in range(index):
            pre = self.nxt[pre]
        self.nxt[pre] = self._new(val, self.nxt[pre])
        self.size += 1

    def deleteAtIndex(self, index):
        if index < 0 or index >= self.size:
            return
        pre = 0
        for _ in range(index):
            pre = self.nxt[pre]
        t = self.nxt[pre]
        self.nxt[pre] = self.nxt[t]
        self.free.append(t)
        self.size -= 1

    def dump(self):
        out, cur = [], self.nxt[0]
        while cur != -1:
            out.append(self.val[cur])
            cur = self.nxt[cur]
        return out


out = run_ops(LinkedArrayPool, ops, args)
print(out)
assert out == EXPECTED

[None, None, None, None, 2, None, 3]


**Code nộp LeetCode (Cách 4):**

```python
class MyLinkedList:
    def __init__(self):
        self.val = [0]
        self.nxt = [-1]
        self.free = []
        self.size = 0

    def _new(self, v: int, nx: int) -> int:
        if self.free:
            s = self.free.pop()
            self.val[s], self.nxt[s] = v, nx
        else:
            self.val.append(v)
            self.nxt.append(nx)
            s = len(self.val) - 1
        return s

    def get(self, index: int) -> int:
        if index < 0 or index >= self.size:
            return -1
        cur = self.nxt[0]
        for _ in range(index):
            cur = self.nxt[cur]
        return self.val[cur]

    def addAtHead(self, val: int) -> None:
        self.addAtIndex(0, val)

    def addAtTail(self, val: int) -> None:
        self.addAtIndex(self.size, val)

    def addAtIndex(self, index: int, val: int) -> None:
        if index > self.size:
            return
        pre = 0
        for _ in range(index):
            pre = self.nxt[pre]
        self.nxt[pre] = self._new(val, self.nxt[pre])
        self.size += 1

    def deleteAtIndex(self, index: int) -> None:
        if index < 0 or index >= self.size:
            return
        pre = 0
        for _ in range(index):
            pre = self.nxt[pre]
        t = self.nxt[pre]
        self.nxt[pre] = self.nxt[t]
        self.free.append(t)
        self.size -= 1
```

**Độ phức tạp:** như Cách 2 (`O(index)` mỗi thao tác); Space `O(số ô tối đa từng dùng)` nhờ tái sử dụng ô.

### Oracle: danh sách Python (chỉ để đối chiếu)

`list` của Python là **mảng động**, không phải danh sách liên kết — đề cấm dùng thư viện danh sách liên kết, nhưng ta dùng `list` làm **chuẩn đối chiếu** hành vi (không nộp).

In [13]:
class ListOracle:
    def __init__(self):
        self.a = []

    def get(self, index):
        return self.a[index] if 0 <= index < len(self.a) else -1

    def addAtHead(self, val):
        self.a.insert(0, val)

    def addAtTail(self, val):
        self.a.append(val)

    def addAtIndex(self, index, val):
        if 0 <= index <= len(self.a):
            self.a.insert(index, val)

    def deleteAtIndex(self, index):
        if 0 <= index < len(self.a):
            del self.a[index]

    def dump(self):
        return list(self.a)


IMPLS = {
    "nodummy": LinkedNoDummy,
    "dummy": LinkedDummy,
    "doubly": LinkedDoubly,
    "pool": LinkedArrayPool,
}
assert run_ops(ListOracle, ops, args) == EXPECTED
for name, cls in IMPLS.items():
    print(f"{name:<8}", run_ops(cls, ops, args))
    assert run_ops(cls, ops, args) == EXPECTED

nodummy  [None, None, None, None, 2, None, 3]
dummy    [None, None, None, None, 2, None, 3]
doubly   [None, None, None, None, 2, None, 3]
pool     [None, None, None, None, 2, None, 3]


## 8. TRACE từng bước

**Trace Cách 2 trên ví dụ** — ta in danh sách và con trỏ `pre` ở mỗi lệnh:

| Lệnh | Đi bao nhiêu bước từ `dummy` | `pre` đứng ở | Hành động | Danh sách sau |
|---|---|---|---|---|
| `addAtHead(1)` | 0 | `dummy` | chèn sau `dummy` | `[1]` |
| `addAtTail(3)` | `cnt = 1` | nút `1` | chèn sau nút `1` | `[1, 3]` |
| `addAtIndex(1, 2)` | 1 | nút `1` | chèn sau nút `1` | `[1, 2, 3]` |
| `get(1)` | 1 (từ `dummy.next`) | — | đọc `val` | trả `2` |
| `deleteAtIndex(1)` | 1 | nút `1` | `pre.next = pre.next.next` | `[1, 3]` |
| `get(1)` | 1 | — | đọc `val` | trả `3` |

In [14]:
def trace_dummy(ops, args):
    obj = LinkedDummy()
    print(f"{'lệnh':<22}{'kết quả':<9}{'cnt':<5}danh sách")
    for op, a in list(zip(ops, args))[1:]:
        res = getattr(obj, op)(*a)
        print(f"{op + str(tuple(a)):<22}{str(res):<9}{obj.cnt:<5}{obj.dump()}")
    return obj


obj = trace_dummy(ops, args)
assert obj.dump() == [1, 3]

lệnh                  kết quả  cnt  danh sách
addAtHead(1,)         None     1    [1]
addAtTail(3,)         None     2    [1, 3]
addAtIndex(1, 2)      None     3    [1, 2, 3]
get(1,)               2        3    [1, 2, 3]
deleteAtIndex(1,)     None     2    [1, 3]
get(1,)               3        2    [1, 3]


**Trace chi tiết một lần `addAtIndex`** — in từng bước đi của `pre`, rồi hai bước nối con trỏ (đúng thứ tự):

In [15]:
def trace_add_at(values, index, val):
    obj = LinkedDummy()
    for v in values:
        obj.addAtTail(v)
    print("trước:", obj.dump(), f"| addAtIndex({index}, {val}), cnt = {obj.cnt}")
    if index > obj.cnt:
        print("index > cnt -> bỏ qua")
        return obj
    pre = obj.dummy
    print("  pre = dummy")
    for step in range(index):
        pre = pre.next
        print(f"  đi bước {step + 1}: pre = nút có val {pre.val}")
    new = ListNode(val)
    new.next = pre.next                                  # bước 1
    print(f"  bước 1: new.next = pre.next  (trỏ vào {new.next.val if new.next else None})")
    pre.next = new                                       # bước 2
    obj.cnt += 1
    print("  bước 2: pre.next = new")
    print("sau  :", obj.dump(), "\n")
    return obj


assert trace_add_at([10, 20, 30], 2, 99).dump() == [10, 20, 99, 30]
assert trace_add_at([10, 20, 30], 3, 99).dump() == [10, 20, 30, 99]       # index == cnt: thêm cuối
assert trace_add_at([10, 20, 30], 4, 99).dump() == [10, 20, 30]           # index > cnt: bỏ qua
assert trace_add_at([], 0, 5).dump() == [5]

trước: [10, 20, 30] | addAtIndex(2, 99), cnt = 3
  pre = dummy
  đi bước 1: pre = nút có val 10
  đi bước 2: pre = nút có val 20
  bước 1: new.next = pre.next  (trỏ vào 30)
  bước 2: pre.next = new
sau  : [10, 20, 99, 30] 

trước: [10, 20, 30] | addAtIndex(3, 99), cnt = 3
  pre = dummy
  đi bước 1: pre = nút có val 10
  đi bước 2: pre = nút có val 20
  đi bước 3: pre = nút có val 30
  bước 1: new.next = pre.next  (trỏ vào None)
  bước 2: pre.next = new
sau  : [10, 20, 30, 99] 

trước: [10, 20, 30] | addAtIndex(4, 99), cnt = 3
index > cnt -> bỏ qua
trước: [] | addAtIndex(0, 5), cnt = 0
  pre = dummy
  bước 1: new.next = pre.next  (trỏ vào None)
  bước 2: pre.next = new
sau  : [5] 



**Trace Cách 3 (danh sách đôi)** — chọn hướng đi cho `get`: với `size = 6`, `index = 1` đi tới từ `head`; `index = 5` đi lùi từ `tail` (chỉ `1` bước thay vì `5`).

In [16]:
def trace_doubly_walk(size, index):
    d = LinkedDoubly()
    for v in range(size):
        d.addAtTail(v * 10)
    if index <= size // 2:
        steps, direction = index, "tới từ head.next"
    else:
        steps, direction = size - index, "lùi từ tail"
    print(f"size={size} index={index}: đi {direction}, {steps} bước -> val {d.get(index)}")
    return steps


assert trace_doubly_walk(6, 1) == 1
assert trace_doubly_walk(6, 5) == 1
assert trace_doubly_walk(6, 3) == 3
assert trace_doubly_walk(7, 6) == 1

size=6 index=1: đi tới từ head.next, 1 bước -> val 10
size=6 index=5: đi lùi từ tail, 1 bước -> val 50
size=6 index=3: đi tới từ head.next, 3 bước -> val 30
size=7 index=6: đi lùi từ tail, 1 bước -> val 60


## 9. Code nộp LeetCode

Ta nộp **Cách 2** (danh sách đơn + `dummy` + `cnt`): ngắn, một đoạn code cho mọi vị trí, đủ nhanh với `≤ 2000` lệnh.

In [17]:
class MyLinkedList:
    def __init__(self):
        self.dummy = ListNode()
        self.cnt = 0

    def get(self, index: int) -> int:
        if index < 0 or index >= self.cnt:
            return -1
        cur = self.dummy.next
        for _ in range(index):
            cur = cur.next
        return cur.val

    def addAtHead(self, val: int) -> None:
        self.addAtIndex(0, val)

    def addAtTail(self, val: int) -> None:
        self.addAtIndex(self.cnt, val)

    def addAtIndex(self, index: int, val: int) -> None:
        if index > self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        pre.next = ListNode(val, pre.next)
        self.cnt += 1

    def deleteAtIndex(self, index: int) -> None:
        if index < 0 or index >= self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        t = pre.next
        pre.next = t.next
        t.next = None
        self.cnt -= 1

    def dump(self):                                    # chỉ dùng để test
        return to_list(self.dummy.next)


# Your MyLinkedList object will be instantiated and called as such:
# obj = MyLinkedList()
# param_1 = obj.get(index)
# obj.addAtHead(val)
# obj.addAtTail(val)
# obj.addAtIndex(index,val)
# obj.deleteAtIndex(index)
out = run_ops(MyLinkedList, ops, args)
print(out)
assert out == EXPECTED
IMPLS["MyLinkedList"] = MyLinkedList

[None, None, None, None, 2, None, 3]


## 10. Vì sao đúng

**Bất biến:** sau mỗi lệnh, (a) chuỗi nút từ `dummy.next` qua `next` có đúng `cnt` nút và kết thúc bằng `None` (không vòng lặp, không đứt), (b) dãy giá trị là dãy mà danh sách trừu tượng (một `list` Python) sẽ có.

- **`get`:** nếu `0 <= index < cnt`, đi `index` bước từ nút đầu tới đúng nút thứ `index` (nút đầu có chỉ số 0); ngoài khoảng đó trả `-1` như đề.
- **`addAtIndex`:** nếu `index > cnt` không làm gì (đề cho phép bỏ qua). Còn lại, đi `index` bước từ `dummy` đến nút `pre` đứng **ngay trước** vị trí `index` (khi `index = 0`, `pre = dummy`; khi `index = cnt`, `pre` là nút cuối thật). Nối `new.next = pre.next` rồi `pre.next = new` đưa nút vào đúng vị trí `index` (trước nút cũ thứ `index`), giữ nguyên thứ tự các nút khác, rồi `cnt += 1`. Thứ tự hai phép gán bảo đảm không mất phần đuôi.
- **`addAtHead` / `addAtTail`:** là `addAtIndex(0, ·)` và `addAtIndex(cnt, ·)` đúng theo định nghĩa.
- **`deleteAtIndex`:** nếu `0 <= index < cnt`, `pre` là nút ngay trước nút cần xóa (có thể là `dummy`); `pre.next = pre.next.next` bỏ nút đó khỏi chuỗi, các nút khác giữ nguyên thứ tự; `cnt -= 1`.
- **Cách 3:** bốn con trỏ `prev/next` của hai nút kề luôn được sửa đồng thời nên hai chiều nhất quán; hai nút canh bảo đảm mọi nút thật đều có `prev` và `next` hợp lệ. Hướng đi (tới hay lùi) chỉ ảnh hưởng số bước, không ảnh hưởng nút tới được.
- **Cách 4:** hệt Cách 2 với "con trỏ = chỉ số ô". Ô trong `free` không còn nằm trên chuỗi từ ô 0 nên không bị đọc nhầm; mỗi ô thuộc hoặc chuỗi hoặc `free` (không cả hai) nên dùng lại an toàn.

Mục 12 đối chiếu cả bốn cách với oracle (một `list` Python) bằng hàng nghìn chuỗi lệnh ngẫu nhiên và còn kiểm tra **cấu trúc nội bộ** (độ dài, không vòng, `prev` nhất quán).

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài danh sách (`≤ 2000`).

| Cách | `get` | `addAtHead` | `addAtTail` | `addAtIndex` / `deleteAtIndex` | Space |
|---|---|---|---|---|---|
| 1. Đơn, không canh | `O(index)` | `O(1)` | `O(n)` | `O(index)` | `O(n)` |
| 2. Đơn + dummy + `cnt` | `O(index)` | `O(1)` | `O(n)` | `O(index)` | `O(n)` |
| 3. Đôi + hai canh | `O(min(index, n − index))` | `O(1)` | `O(1)` | `O(min(index, n − index))` | `O(n)` (thêm `prev`) |
| 4. Trên mảng | `O(index)` | `O(1)` | `O(n)` | `O(index)` | `O(n)`, ô được tái sử dụng |

Với `≤ 2000` lệnh và `n ≤ 2000` mọi cách đều tức thì. Mục 12.7 thử vượt ràng buộc để thấy `addAtTail` của Cách 3 không bị chậm theo `n`, còn các cách còn lại tăng bậc hai khi thêm liên tiếp.

## 12. Bộ test

Chiến lược:

1. **Ví dụ đề** qua `run_ops` cho 4 cách + `MyLinkedList` (so với kết quả đề).
2. **Edge case:** thao tác trên danh sách rỗng (`get`, `deleteAtIndex`, `addAtIndex` với `index > 0`), `index == độ dài`, `index > độ dài`, xóa đầu/cuối/duy nhất, `get` ở biên `0` và `cnt − 1`, `cnt`.
3. **Cấu trúc nội bộ:** hàm `check_structure` đi dọc danh sách kiểm tra `độ dài == cnt`, không vòng, kết thúc đúng, và (Cách 3) `prev` nhất quán với `next`, (Cách 4) ô không bị lẫn giữa chuỗi và `free`.
4. **Ngẫu nhiên có seed:** các chuỗi lệnh đúng ràng buộc (`≤ 2000` lệnh, `0 <= index, val <= 1000`) **so với oracle** sau từng lệnh.
5. **Tăng trưởng lớn:** thêm rồi xóa hàng trăm phần tử (độ dài `> 1000`).
6. **Đối chiếu chéo** các cách với nhau trên cùng chuỗi lệnh.
7. **Thời gian** (vượt ràng buộc).

In [18]:
def check_structure(obj, expected):
    """Kiểm tra cấu trúc nội bộ của từng cài đặt khớp danh sách mong đợi."""
    assert obj.dump() == expected, (obj.dump(), expected)
    if isinstance(obj, (LinkedDummy, MyLinkedList)):
        assert obj.cnt == len(expected)
        seen, cur = set(), obj.dummy.next
        while cur is not None:
            assert id(cur) not in seen, "có vòng lặp"
            seen.add(id(cur))
            cur = cur.next
        assert len(seen) == len(expected)
    elif isinstance(obj, LinkedNoDummy):
        assert obj.size == len(expected)
    elif isinstance(obj, LinkedDoubly):
        assert obj.size == len(expected)
        cur, prev, count = obj.head.next, obj.head, 0
        while cur is not obj.tail:
            assert cur.prev is prev, "prev không nhất quán"
            prev, cur, count = cur, cur.next, count + 1
            assert count <= len(expected)
        assert obj.tail.prev is prev and count == len(expected)
    elif isinstance(obj, LinkedArrayPool):
        assert obj.size == len(expected)
        chain, cur = [], obj.nxt[0]
        while cur != -1:
            chain.append(cur)
            cur = obj.nxt[cur]
            assert len(chain) <= len(obj.val), "có vòng lặp"
        assert not set(chain) & set(obj.free), "ô vừa trong chuỗi vừa trong free"
        assert len(set(chain)) == len(chain)
        assert len(chain) + len(obj.free) == len(obj.val) - 1, "rò ô nhớ"


# 1) Ví dụ đề
for name, cls in IMPLS.items():
    assert run_ops(cls, ops, args) == EXPECTED, name
print("Ví dụ đề (4 cách + MyLinkedList): OK")

# 2) Edge case
def script(cls, steps):
    obj = cls()
    out, ref = [], ListOracle()
    for op, a in steps:
        r = getattr(obj, op)(*a)
        e = getattr(ref, op)(*a)
        assert r == e, (cls.__name__, op, a, r, e)
        out.append(r)
        check_structure(obj, ref.dump())
    return out


edge = [
    ("get", (0,)), ("get", (5,)), ("deleteAtIndex", (0,)), ("deleteAtIndex", (3,)),     # rỗng
    ("addAtIndex", (1, 7)), ("get", (0,)),                                              # index > cnt (=0): bỏ qua
    ("addAtIndex", (0, 7)), ("get", (0,)), ("get", (1,)),                               # index == cnt (=0): thêm
    ("addAtIndex", (1, 8)), ("addAtIndex", (3, 9)), ("addAtIndex", (2, 9)),             # =cnt hợp lệ, >cnt bỏ qua, =cnt
    ("get", (2,)), ("get", (3,)), ("get", (1000,)),
    ("deleteAtIndex", (2,)), ("deleteAtIndex", (2,)),                                   # xóa cuối rồi index == cnt: bỏ qua
    ("deleteAtIndex", (0,)), ("deleteAtIndex", (0,)), ("deleteAtIndex", (0,)),          # xóa đầu đến rỗng
    ("addAtTail", (5,)), ("addAtHead", (4,)), ("addAtTail", (6,)), ("addAtIndex", (3, 1000)),
    ("get", (3,)), ("get", (0,)),
]
for name, cls in IMPLS.items():
    script(cls, edge)
print("Edge case: OK (", len(edge), "lệnh x", len(IMPLS), "cài đặt, so với oracle + kiểm tra cấu trúc )")

Ví dụ đề (4 cách + MyLinkedList): OK
Edge case: OK ( 26 lệnh x 5 cài đặt, so với oracle + kiểm tra cấu trúc )


In [19]:
# 4) Ngẫu nhiên có seed: chuỗi lệnh đúng ràng buộc, so với oracle SAU TỪNG LỆNH
def gen_steps(rng, calls, grow=0.6):
    steps, length = [], 0
    for _ in range(calls):
        c = rng.random()
        val = rng.randint(0, 1000)
        if c < 0.12:
            steps.append(("addAtHead", (val,))); length += 1
        elif c < 0.24:
            steps.append(("addAtTail", (val,))); length += 1
        elif c < 0.24 + 0.24 * grow:
            idx = rng.randint(0, length + 1) if rng.random() < 0.9 else rng.randint(0, 1000)
            steps.append(("addAtIndex", (idx, val)))
            length += idx <= length
        elif c < 0.75:
            idx = rng.randint(0, length + 1) if rng.random() < 0.9 else rng.randint(0, 1000)
            steps.append(("get", (idx,)))
        else:
            idx = rng.randint(0, length + 1) if rng.random() < 0.9 else rng.randint(0, 1000)
            steps.append(("deleteAtIndex", (idx,)))
            length -= idx < length
    return steps


rng = random.Random(707)
total_ops = 0
for t in range(120):
    steps = gen_steps(rng, rng.randint(1, 300), rng.choice([0.2, 0.6, 1.0]))
    for name, cls in IMPLS.items():
        script(cls, steps)
    total_ops += len(steps)
print(f"Ngẫu nhiên: 120 chuỗi, {total_ops} lệnh x {len(IMPLS)} cài đặt, kiểm tra cấu trúc sau từng lệnh: OK")

# 5) Một chuỗi dài đúng ràng buộc 2000 lệnh và độ dài lớn
steps = [("addAtTail", (i % 1001,)) for i in range(1200)]
steps += [("get", (i,)) for i in range(0, 1300, 50)]
steps += [("deleteAtIndex", (0,)) for _ in range(300)] + [("addAtIndex", (i, i)) for i in range(0, 500, 5)]
assert len(steps) <= 2000
for name, cls in IMPLS.items():
    obj = cls()
    ref = ListOracle()
    for op, a in steps:
        assert getattr(obj, op)(*a) == getattr(ref, op)(*a)
    check_structure(obj, ref.dump())
print(f"Chuỗi dài {len(steps)} lệnh (độ dài tối đa 1200): OK")

Ngẫu nhiên: 120 chuỗi, 17132 lệnh x 5 cài đặt, kiểm tra cấu trúc sau từng lệnh: OK
Chuỗi dài 1626 lệnh (độ dài tối đa 1200): OK


In [20]:
# 6) Đối chiếu chéo + kiểm tra tái sử dụng ô nhớ của Cách 4
rng = random.Random(77)
for _ in range(100):
    steps = gen_steps(rng, 300, 0.8)
    ops_ = ["MyLinkedList"] + [s[0] for s in steps]
    args_ = [[]] + [list(s[1]) for s in steps]
    outs = [run_ops(cls, ops_, args_) for cls in IMPLS.values()]
    assert all(o == outs[0] for o in outs)
    assert outs[0] == run_ops(ListOracle, ops_, args_)

pool = LinkedArrayPool()
for rep in range(50):                                 # thêm 100, xóa 100, lặp lại: mảng không được phình ra
    for i in range(100):
        pool.addAtTail(i)
    for i in range(100):
        pool.deleteAtIndex(0)
assert len(pool.val) == 101 and pool.size == 0
print("Đối chiếu chéo 100 chuỗi: OK; Cách 4 tái sử dụng ô (mảng chỉ dài", len(pool.val), "sau 5000 thêm + 5000 xóa)")

Đối chiếu chéo 100 chuỗi: OK; Cách 4 tái sử dụng ô (mảng chỉ dài 101 sau 5000 thêm + 5000 xóa)


In [21]:
# 7) Thời gian (VƯỢT ràng buộc): thêm liên tiếp ở cuối rồi get ngẫu nhiên
def bench(cls, n):
    obj = cls()
    t0 = time.perf_counter()
    for i in range(n):
        obj.addAtTail(i)
    t_add = (time.perf_counter() - t0) * 1000
    rng = random.Random(1)
    t0 = time.perf_counter()
    s = 0
    for _ in range(200):
        s += obj.get(rng.randrange(n))
    t_get = (time.perf_counter() - t0) * 1000
    return t_add, t_get, s


for n in (1000, 2000, 4000):
    row, sums = [], set()
    for name in ("nodummy", "dummy", "doubly", "pool"):
        ta, tg, s = bench(IMPLS[name], n)
        sums.add(s)
        row.append(f"{name} add {ta:7.1f} / get {tg:6.1f} ms")
    assert len(sums) == 1
    print(f"n={n:>5}: " + " | ".join(row))

n= 1000: nodummy add     8.2 / get    1.8 ms | dummy add     7.9 / get    1.8 ms | doubly add     0.3 / get    0.8 ms | pool add    10.2 / get    2.5 ms
n= 2000: nodummy add    36.5 / get    3.7 ms | dummy add    40.2 / get    3.9 ms | doubly add     0.7 / get    1.8 ms | pool add    49.2 / get    6.1 ms


n= 4000: nodummy add   141.6 / get    8.0 ms | dummy add   139.8 / get    8.2 ms | doubly add     1.3 / get    3.5 ms | pool add   169.5 / get    9.4 ms


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên cập nhật độ dài `cnt`

Nếu `addAtIndex` quên `cnt += 1`, `get` và các kiểm tra biên dùng `cnt` cũ ⇒ trả `-1` cho nút có thật và chặn thêm đuôi.

In [22]:
class WrongNoCount(LinkedDummy):
    def addAtIndex(self, index, val):
        if index > self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        pre.next = ListNode(val, pre.next)           # sai: thiếu self.cnt += 1


out = run_ops(WrongNoCount, ops, args)
print("sai :", out, "| đúng:", run_ops(MyLinkedList, ops, args))
assert out != EXPECTED

sai : [None, None, None, None, -1, None, -1] | đúng: [None, None, None, None, 2, None, 3]


### Lỗi 2: nối con trỏ **sai thứ tự**

`pre.next = new` trước rồi `new.next = pre.next` làm `new.next` trỏ **chính nó**: phần đuôi bị mất và danh sách thành vòng lặp.

In [23]:
class WrongLinkOrder(LinkedDummy):
    def addAtIndex(self, index, val):
        if index > self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        node = ListNode(val)
        pre.next = node                              # sai: nối trước
        node.next = pre.next                         # node.next = node  -> vòng lặp tự thân
        self.cnt += 1


obj = WrongLinkOrder()
obj.addAtHead(1)
obj.addAtHead(2)
print("nút đầu trỏ chính nó?", obj.dummy.next.next is obj.dummy.next)
assert obj.dummy.next.next is obj.dummy.next          # vòng lặp: get(1) sẽ đi mãi trong vòng
assert obj.get(0) == 2

nút đầu trỏ chính nó? True


### Lỗi 3: `>=` thay cho `>` khi kiểm tra `index` của `addAtIndex`

`addAtIndex(cnt, val)` **hợp lệ** (thêm vào cuối). Nếu viết `if index >= self.cnt: return` thì `addAtTail` (gọi `addAtIndex(cnt, ...)`) không bao giờ thêm được.

In [24]:
class WrongBoundary(LinkedDummy):
    def addAtIndex(self, index, val):
        if index >= self.cnt:                        # sai: phải là index > self.cnt
            return
        super().addAtIndex(index, val)


w = WrongBoundary()
w.addAtHead(1)                                       # cnt=0, index=0 >= 0 -> bị bỏ qua
w.addAtTail(3)
print("sai :", w.dump(), "| đúng: [1, 3]")
assert w.dump() == []

sai : [] | đúng: [1, 3]


### Lỗi 4: không có `dummy` mà quên xử lý **xóa/ thêm ở đầu**

Không có nút canh, `pre` của vị trí `0` không tồn tại. Cố đi `index − 1 = −1` bước (vòng `range(-1)` bỏ qua) rồi dùng `pre = head` sẽ xóa **nút thứ 1** thay vì nút thứ 0.

In [25]:
class WrongNoHeadCase(LinkedNoDummy):
    def deleteAtIndex(self, index):
        if index < 0 or index >= self.size:
            return
        pre = self.head
        for _ in range(index - 1):                   # index = 0 -> range(-1) rỗng, pre = head
            pre = pre.next
        pre.next = pre.next.next                     # xóa nút THỨ 1 (hoặc lỗi khi chỉ có 1 nút)
        self.size -= 1


w = WrongNoHeadCase()
for v in (10, 20, 30):
    w.addAtTail(v)
w.deleteAtIndex(0)
print("sai :", w.dump(), "(đáng lẽ [20, 30])")
assert w.dump() == [10, 30]
w2 = WrongNoHeadCase()
w2.addAtHead(5)
try:
    w2.deleteAtIndex(0)
except AttributeError as e:
    print("một nút duy nhất -> AttributeError:", e)

sai : [10, 30] (đáng lẽ [20, 30])
một nút duy nhất -> AttributeError: 'NoneType' object has no attribute 'next'


### Lỗi 5 (danh sách đôi): quên sửa `prev`

Chèn mà chỉ sửa `next` (quên `nxt.prev = node`) thì hai chiều **lệch nhau**: lần chèn kế tiếp đọc `prev` cũ của nút sau nên **làm mất nút**, và `get` ở nửa sau (đi lùi bằng `prev`) trả sai.

In [26]:
class WrongNoPrev(LinkedDoubly):
    def addAtIndex(self, index, val):
        if index > self.size:
            return
        nxt = self._node_at(index)
        pre = nxt.prev
        node = DNode(val, pre, nxt)
        pre.next = node                              # sai: thiếu nxt.prev = node
        self.size += 1


w = WrongNoPrev()
for v in (1, 2, 3, 4):
    w.addAtTail(v)
print("duyệt tới (đáng lẽ [1, 2, 3, 4]):", w.dump())
try:
    res = (w.get(3), w.get(2))
    print("get(3), get(2):", res, "| đúng: (4, 3)")
except AttributeError as e:
    res = None
    print("get(3) đi lùi qua prev hỏng -> AttributeError:", e)
good = LinkedDoubly()
for v in (1, 2, 3, 4):
    good.addAtTail(v)
assert good.get(3) == 4 and good.get(2) == 3
assert res != (4, 3)

duyệt tới (đáng lẽ [1, 2, 3, 4]): [4]
get(3) đi lùi qua prev hỏng -> AttributeError: 'NoneType' object has no attribute 'val'


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** "tự cài cấu trúc dữ liệu với các thao tác chèn/xóa/truy cập" ⇒ liệt kê **bất biến** (độ dài, không vòng, thứ tự) rồi mỗi thao tác chỉ cần giữ bất biến.
- **Luôn dùng nút canh `dummy`** cho danh sách đơn: bỏ hẳn các trường hợp đặc biệt ở đầu. Nhớ nó là mẹo "chuẩn" của cả họ bài danh sách liên kết (*Remove Linked List Elements*, *Merge Two Sorted Lists*, *Partition List*...).
- **Hai biên khác nhau:** `get/delete` cần `index < cnt`; `add` cần `index <= cnt`. Viết ra giấy khi mới đọc đề.
- **Thứ tự nối:** gán `new.next = pre.next` **trước**, `pre.next = new` **sau** (nhớ "nối đuôi trước, nối đầu sau").
- **Danh sách đôi** khi cần thao tác ở **cả hai đầu** hoặc xóa nút khi đã có tham chiếu tới nó (như *LRU Cache*, *LFU Cache*); đi từ đầu gần hơn nửa số bước.
- **`cnt` thay vì đi hết danh sách** để biết độ dài.
- **Test bằng oracle:** dùng `list` Python làm chuẩn rồi so sánh sau từng lệnh — rất hiệu quả cho bài thiết kế.
- **Cùng họ:** *LRU Cache* (146), *Design Browser History* (1472), *Design Circular Deque* (641), *Design Skiplist* (1206), *Flatten a Multilevel Doubly Linked List* (430).

## 15. Biến thể của bài

1. **Giữ thêm con trỏ `tail`** để `addAtTail` còn `O(1)` ngay cả với danh sách đơn (cẩn thận cập nhật `tail` khi xóa nút cuối hoặc chèn vào cuối).
2. **Thêm `reverse()`** đảo ngược danh sách tại chỗ (bài tập 5).
3. **Thêm `size()`/`indexOf(val)`/`removeValue(val)`.**
4. **Danh sách vòng** (circular): nút cuối trỏ về nút đầu (xem bài 708).
5. **Skip list** để truy cập `O(log n)`.
6. **Hỗ trợ chỉ số âm** kiểu Python (`get(-1)` là nút cuối) hoặc chỉ số âm được xem là `0` (phiên bản cũ của đề).

Cài nhanh biến thể 1 (có `tail`) và kiểm tra bằng đúng bộ oracle ở mục 12:

In [27]:
class LinkedWithTail:
    """Danh sách đơn + dummy + cnt + tail: addAtTail O(1)."""

    def __init__(self):
        self.dummy = ListNode()
        self.tail = self.dummy                         # khi rỗng, tail là dummy
        self.cnt = 0

    def get(self, index):
        if index < 0 or index >= self.cnt:
            return -1
        cur = self.dummy.next
        for _ in range(index):
            cur = cur.next
        return cur.val

    def addAtHead(self, val):
        self.addAtIndex(0, val)

    def addAtTail(self, val):
        self.tail.next = ListNode(val)
        self.tail = self.tail.next
        self.cnt += 1

    def addAtIndex(self, index, val):
        if index > self.cnt:
            return
        if index == self.cnt:
            self.addAtTail(val)
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        pre.next = ListNode(val, pre.next)
        self.cnt += 1

    def deleteAtIndex(self, index):
        if index < 0 or index >= self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        if pre.next is self.tail:                      # xóa nút cuối: tail lùi về pre
            self.tail = pre
        pre.next = pre.next.next
        self.cnt -= 1

    def dump(self):
        return to_list(self.dummy.next)


rng = random.Random(15)
for _ in range(200):
    steps = gen_steps(rng, 300, 0.8)
    obj, ref = LinkedWithTail(), ListOracle()
    for op, a in steps:
        assert getattr(obj, op)(*a) == getattr(ref, op)(*a)
    assert obj.dump() == ref.dump()
    last = obj.dummy
    while last.next:
        last = last.next
    assert obj.tail is last, "tail phải luôn là nút cuối thật"
n = 4000
t0 = time.perf_counter(); o = LinkedWithTail()
for i in range(n):
    o.addAtTail(i)
print(f"tail-pointer: {n} lần addAtTail mất {(time.perf_counter() - t0) * 1000:.1f} ms (Cách 2 ở mục 12.7: bậc hai)")
print("Biến thể có tail khớp oracle trên 200 chuỗi: OK")

tail-pointer: 4000 lần addAtTail mất 1.3 ms (Cách 2 ở mục 12.7: bậc hai)
Biến thể có tail khớp oracle trên 200 chuỗi: OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | `get` / `add` / `delete` giữa | `addAtTail` | Space | Khi nào dùng |
|---|---|---|---|---|---|
| 1. Đơn, không canh | nhánh riêng cho đầu danh sách | `O(index)` | `O(n)` | `O(n)` | để thấy vì sao cần `dummy` |
| 2. Đơn + `dummy` + `cnt` | một đoạn code cho mọi vị trí | `O(index)` | `O(n)` | `O(n)` | **làm bài** |
| 3. Đôi + hai canh | `prev` + đi từ đầu gần hơn | `O(min(i, n−i))` | `O(1)` | `O(n)` (+`prev`) | thao tác hai đầu thường xuyên, hoặc khi có sẵn tham chiếu nút |
| 4. Trên mảng | con trỏ = chỉ số ô + `free` | `O(index)` | `O(n)` | `O(n)` | ngôn ngữ không có tham chiếu; bộ nhớ liền kề; tái sử dụng ô |

Trực giác chọn: đơn + `dummy` là mặc định; muốn `addAtTail` `O(1)` thì thêm `tail` (mục 15) hoặc dùng danh sách đôi.

## 17. 5 bài tập vận dụng (dễ → khó)

Mọi cell TODO **không gọi** hàm khi chạy notebook; đáp án có assert ở mục 19.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

```python
l = MyLinkedList()
l.addAtHead(5); l.addAtTail(6); l.addAtIndex(5, 9); l.addAtIndex(2, 7)
print(l.dump(), l.get(2), l.get(3))
l.deleteAtIndex(3); l.deleteAtIndex(0)
print(l.dump(), l.get(0))
```

In [28]:
l_b1 = MyLinkedList()
l_b1.addAtHead(5); l_b1.addAtTail(6); l_b1.addAtIndex(5, 9); l_b1.addAtIndex(2, 7)
print(l_b1.dump(), l_b1.get(2), l_b1.get(3))
l_b1.deleteAtIndex(3); l_b1.deleteAtIndex(0)
print(l_b1.dump(), l_b1.get(0))

[5, 6, 7] 7 -1
[6, 7] 6


### Bài 2 (tính tay)

Bắt đầu với danh sách rỗng. Thực hiện lần lượt: `addAtHead(3)`, `addAtHead(2)`, `addAtTail(4)`, `addAtIndex(1, 9)`, `deleteAtIndex(2)`, `addAtIndex(4, 8)`, `addAtIndex(3, 8)`. Danh sách cuối cùng (từ chỉ số 0) là gì? Ghi vào `my_answer_2` (kiểu `list`; để `None` đến khi tính xong).

In [29]:
steps_b2 = [("addAtHead", (3,)), ("addAtHead", (2,)), ("addAtTail", (4,)), ("addAtIndex", (1, 9)),
            ("deleteAtIndex", (2,)), ("addAtIndex", (4, 8)), ("addAtIndex", (3, 8))]
my_answer_2 = None          # TODO: điền list tính tay, ví dụ [1, 2, 3]

### Bài 3 (điền TODO)

Hoàn thành `deleteAtIndex` cho lớp danh sách đơn có `dummy` dưới đây (đi tới nút `pre`, nối `pre.next` qua nút bị xóa, cập nhật `cnt`). Cell chỉ **định nghĩa**, không gọi.

In [30]:
class LinkedTodo:
    def __init__(self):
        self.dummy = ListNode()
        self.cnt = 0

    def addAtTail(self, val):
        pre = self.dummy
        for _ in range(self.cnt):
            pre = pre.next
        pre.next = ListNode(val)
        self.cnt += 1

    def deleteAtIndex(self, index):
        # TODO: kiểm tra biên; đi index bước từ dummy tới pre; pre.next = pre.next.next; giảm cnt
        pass

    def dump(self):
        return to_list(self.dummy.next)

### Bài 4 (tự viết lại)

Tự viết lại **Cách 3** (danh sách đôi hai nút canh): lớp `MyDoubly` với đủ 5 thao tác. Không nhìn code cũ. Cell dưới chỉ để khung rỗng.

In [31]:
class MyDoubly:
    # TODO: viết __init__, get, addAtHead, addAtTail, addAtIndex, deleteAtIndex (+ dump để test)
    pass

### Bài 5 (biến thể / mini challenge)

Thêm phương thức `reverse()` **đảo ngược danh sách tại chỗ** cho lớp có `dummy` + `cnt` (không tạo nút mới; chỉ đổi `next`). Gợi ý: dùng ba biến `prev, cur, nxt` rồi cuối cùng đặt `dummy.next = prev`.

In [32]:
class ReversibleList(LinkedDummy):
    def reverse(self):
        # TODO: đảo chiều các con trỏ next; cuối cùng dummy.next = prev
        pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: `addAtIndex(5, 9)` có `index > cnt` không?
- Bài 2: vẽ danh sách sau từng lệnh; nhớ `index == cnt` là hợp lệ, `index > cnt` thì bỏ qua.
- Bài 3: `get` và `deleteAtIndex` hợp lệ khi `0 <= index < cnt`.
- Bài 4: hai nút canh, `head.next = tail`, `tail.prev = head`.
- Bài 5: `prev = None`, `cur = dummy.next`; lặp: lưu `nxt = cur.next`, đổi `cur.next = prev`, rồi tiến lên.

**Mức 2 (vừa):**
- Bài 1: sau hai lệnh đầu danh sách `[5, 6]`; `addAtIndex(5, 9)` bị bỏ qua (5 > 2); `addAtIndex(2, 7)` thêm vào cuối.
- Bài 2: `[2, 3]` → `[2, 3, 4]` → chèn 9 vào chỉ số 1 → xóa chỉ số 2 → thử `addAtIndex(4, 8)` (độ dài hiện tại?) → `addAtIndex(3, 8)`.
- Bài 3: `pre = self.dummy; for _ in range(index): pre = pre.next; pre.next = pre.next.next; self.cnt -= 1`.
- Bài 4: `_node_at(index)` đi tới hoặc lùi; thêm: sửa 4 con trỏ; xóa: sửa 2 con trỏ.
- Bài 5: vòng lặp `while cur:`; `prev, cur = cur, nxt` ở cuối mỗi vòng.

**Mức 3 (gần đáp án):**
- Bài 1: dòng 1 in `[5, 6, 7] 7 -1`; dòng 2 (sau xóa chỉ số 3: bỏ qua, xóa chỉ số 0: `5`) in `[6, 7] 6`.
- Bài 2: sau `addAtIndex(1, 9)`: `[2, 9, 3, 4]`; xóa chỉ số 2 → `[2, 9, 4]`; `addAtIndex(4, 8)` bỏ qua (4 > 3); `addAtIndex(3, 8)` → `[2, 9, 4, 8]`.
- Bài 3: nhớ `t = pre.next; pre.next = t.next`.
- Bài 4: trong `addAtIndex`: `node = DNode(val, pre, nxt); pre.next = node; nxt.prev = node`.
- Bài 5: ba dòng trong vòng lặp: `nxt = cur.next; cur.next = prev; prev, cur = cur, nxt`.

## 19. Đáp án tham khảo

Phần này đặt cuối bài giảng; mọi đáp án có assert.

In [33]:
# Bài 1
assert l_b1.dump() == [6, 7] and l_b1.get(0) == 6
chk = MyLinkedList()
chk.addAtHead(5); chk.addAtTail(6); chk.addAtIndex(5, 9); chk.addAtIndex(2, 7)
assert chk.dump() == [5, 6, 7] and chk.get(2) == 7 and chk.get(3) == -1
print("Bài 1: [5, 6, 7] 7 -1  /  [6, 7] 6")

# Bài 2
ans_2 = [2, 9, 4, 8]
ref = ListOracle()
for op, a in steps_b2:
    getattr(ref, op)(*a)
assert ref.dump() == ans_2
for cls in IMPLS.values():
    o = cls()
    for op, a in steps_b2:
        getattr(o, op)(*a)
    assert o.dump() == ans_2
print("Bài 2:", ans_2)


# Bài 3
class LinkedTodoSol(LinkedTodo):
    def deleteAtIndex(self, index):
        if index < 0 or index >= self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        t = pre.next
        pre.next = t.next
        self.cnt -= 1


rng = random.Random(3)
for _ in range(100):
    o, ref = LinkedTodoSol(), []
    for _ in range(40):
        if rng.random() < 0.6:
            v = rng.randint(0, 1000); o.addAtTail(v); ref.append(v)
        else:
            i = rng.randint(0, len(ref) + 1); o.deleteAtIndex(i)
            if i < len(ref):
                del ref[i]
        assert o.dump() == ref and o.cnt == len(ref)
print("Bài 3: OK")

Bài 1: [5, 6, 7] 7 -1  /  [6, 7] 6
Bài 2: [2, 9, 4, 8]
Bài 3: OK


In [34]:
# Bài 4
class DNodeX:
    def __init__(self, val=0, prev=None, next=None):
        self.val, self.prev, self.next = val, prev, next


class MyDoublySol:
    def __init__(self):
        self.head, self.tail = DNodeX(), DNodeX()
        self.head.next, self.tail.prev = self.tail, self.head
        self.size = 0

    def _node_at(self, index):
        if index <= self.size // 2:
            cur = self.head.next
            for _ in range(index):
                cur = cur.next
        else:
            cur = self.tail
            for _ in range(self.size - index):
                cur = cur.prev
        return cur

    def get(self, index):
        return -1 if index < 0 or index >= self.size else self._node_at(index).val

    def addAtHead(self, val):
        self.addAtIndex(0, val)

    def addAtTail(self, val):
        self.addAtIndex(self.size, val)

    def addAtIndex(self, index, val):
        if index > self.size:
            return
        nxt = self._node_at(index)
        pre = nxt.prev
        node = DNodeX(val, pre, nxt)
        pre.next = nxt.prev = node
        self.size += 1

    def deleteAtIndex(self, index):
        if index < 0 or index >= self.size:
            return
        node = self._node_at(index)
        node.prev.next, node.next.prev = node.next, node.prev
        self.size -= 1

    def dump(self):
        out, cur = [], self.head.next
        while cur is not self.tail:
            out.append(cur.val)
            cur = cur.next
        return out


assert run_ops(MyDoublySol, ops, args) == EXPECTED
rng = random.Random(4)
for _ in range(100):
    steps = gen_steps(rng, 200, 0.8)
    o, ref = MyDoublySol(), ListOracle()
    for op, a in steps:
        assert getattr(o, op)(*a) == getattr(ref, op)(*a)
    assert o.dump() == ref.dump()
print("Bài 4: OK")


# Bài 5
class ReversibleListSol(LinkedDummy):
    def reverse(self):
        prev, cur = None, self.dummy.next
        while cur:
            nxt = cur.next
            cur.next = prev
            prev, cur = cur, nxt
        self.dummy.next = prev


rng = random.Random(5)
for _ in range(100):
    o = ReversibleListSol()
    vals = [rng.randint(0, 1000) for _ in range(rng.randint(0, 30))]
    for v in vals:
        o.addAtTail(v)
    o.reverse()
    assert o.dump() == vals[::-1] and o.cnt == len(vals)
    o.addAtTail(1)                          # vẫn dùng được sau khi đảo
    assert o.get(len(vals)) == 1
print("Bài 5: OK")

Bài 4: OK


Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template (đơn + `dummy` + `cnt`):**

```python
class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next


class MyLinkedList:
    def __init__(self):
        self.dummy, self.cnt = ListNode(), 0

    def get(self, index):
        if not 0 <= index < self.cnt:
            return -1
        cur = self.dummy.next
        for _ in range(index):
            cur = cur.next
        return cur.val

    def addAtHead(self, val):  self.addAtIndex(0, val)
    def addAtTail(self, val):  self.addAtIndex(self.cnt, val)

    def addAtIndex(self, index, val):
        if index > self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        pre.next = ListNode(val, pre.next)
        self.cnt += 1

    def deleteAtIndex(self, index):
        if not 0 <= index < self.cnt:
            return
        pre = self.dummy
        for _ in range(index):
            pre = pre.next
        pre.next = pre.next.next
        self.cnt -= 1
```

**Công thức nhớ:** *đi `index` bước từ `dummy` để có `pre`; chèn: `new.next = pre.next; pre.next = new`; xóa: `pre.next = pre.next.next`; thêm cần `index <= cnt`, get/xóa cần `index < cnt`.*

**Checklist trước khi submit:**

- [ ] Tự định nghĩa lớp nút (skeleton LeetCode không cho sẵn).
- [ ] Có nút `dummy` (hoặc hai nút canh) để không phải xử lý riêng đầu danh sách.
- [ ] Cập nhật `cnt`/`size` ở **mọi** thao tác thêm và xóa.
- [ ] Biên: `get/delete` dùng `index < cnt`; `add` dùng `index <= cnt`; ngoài khoảng thì bỏ qua (`get` trả `-1`).
- [ ] Thứ tự nối: `new.next = pre.next` **trước**, `pre.next = new` **sau**.
- [ ] Danh sách đôi: sửa cả `prev` lẫn `next` của hai nút kề.
- [ ] Đã test: rỗng, `index == cnt`, `index > cnt`, xóa đầu/cuối/duy nhất, nhiều thao tác ngẫu nhiên so với `list`.
- [ ] Không dùng thư viện danh sách liên kết có sẵn.